# Chapter 1: Basic Prompt Structure

**Technique:** Prompt anatomy with system, user, and assistant roles

**Contents**
* [Lesson: The Messages API](#lesson)
* [Exercises](#exercises)
* [Example Playground](#example-playground)

In [ ]:
# Make the project root importable from any working directory, then load the helpers.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "lib" / "helpers.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lib.helpers import get_completion, response_text, client, MODEL, MAX_TOKENS, SAMPLING_PARAMS

<a id="lesson"></a>
## Lesson: The Messages API

Every Claude API call requires three top level fields:

* `model` (`str`): pinned model ID (e.g. `claude-opus-5-5`)
* `max_tokens` (`int`): hard upper bound on output tokens
* `messages` (`list`): ordered list of conversation turns

Each element of `messages` is a dict `{"role": ..., "content": ...}` where `role` is either `"user"` or `"assistant"`. A well-formed conversation follows two conventions:

1. **The first message is from the `user`.**
2. **Roles alternate**: `user`, `assistant`, `user`, ... one turn at a time.

The API itself enforces less than that, and it pays to know exactly what:

* **Every message needs a `role`.** A message without one is rejected with a `400`.
* **The conversation must end with a `user` turn** (on current models). A trailing `assistant` turn, the old "prefill" pattern, is rejected with a `400`.
* **Consecutive turns with the same role are merged**, and a leading `assistant` turn is accepted. These no longer fail loudly, which is exactly why you should get them right in your own code: a bug in your conversation-state logic silently changes what Claude sees instead of raising an error.

The `system` prompt is a separate top level string, not a message turn. Use it to set Claude's persona, constraints, or output format. The helper `get_completion(prompt, system="")` wires all of this up; `system` defaults to `""` (no system prompt).

When you call `client.messages.create` directly, as some cells below do, we also pass `**SAMPLING_PARAMS`. It adds `temperature=0` on older models that accept it and nothing on current models (see Chapter 0).

This is the same shape you'll wire into a backend service: construct a `messages` list from your application's conversation state, pass it to the API, stream or wait for the reply.

In [ ]:
print(get_completion("What is the time complexity of binary search?"))

In [ ]:
print(get_completion("What year was Python first released?"))

### Malformed requests (intentional errors)

The cells below send requests that break the message rules. Two of them violate the API contract and **are rejected with a `400 Bad Request`**; we catch the `anthropic.BadRequestError` and print it so you can read exactly what the API enforces (and so "Run All" keeps going). The middle one shows how the API treats consecutive turns with the same role.

In [ ]:
import anthropic

try:
    res = client.messages.create(
        model=MODEL,
        max_tokens=2000,
        messages=[{"content": "Hi Claude, how are you?"}],  # missing role: the API rejects this
        **SAMPLING_PARAMS,
    )
    print(response_text(res))
except anthropic.BadRequestError as err:
    print("400 Bad Request:", err.message)

In [ ]:
# Two consecutive "user" turns. Older versions of the API rejected this with a 400;
# the current API merges consecutive same-role messages into a single user turn,
# so Claude answers both questions. Build alternating turns in your own code anyway.
try:
    res = client.messages.create(
        model=MODEL,
        max_tokens=2000,
        messages=[
            {"role": "user", "content": "What is REST?"},
            {"role": "user", "content": "Also, what is gRPC?"},
        ],
        **SAMPLING_PARAMS,
    )
    print(response_text(res))
except anthropic.BadRequestError as err:
    print("400 Bad Request:", err.message)

In [ ]:
# A conversation that ends with an assistant turn (a "prefill") is rejected on
# current models: the last message must be the user turn you want answered.
try:
    res = client.messages.create(
        model=MODEL,
        max_tokens=2000,
        messages=[
            {"role": "user", "content": "Classify this ticket: 'I was charged twice this month.'"},
            {"role": "assistant", "content": "Category:"},  # trailing assistant turn
        ],
        **SAMPLING_PARAMS,
    )
    print(response_text(res))
except anthropic.BadRequestError as err:
    print("400 Bad Request:", err.message)

### System prompts

The `system` prompt is delivered to Claude before any user turn. Use it to supply context and rules that should govern the entire conversation: persona, output format, domain constraints, safety guardrails. The user turn then carries the actual request.

The example below configures Claude as a requirements reviewer that only asks clarifying questions, never proposes solutions. This pattern is useful for guardrailing Claude to a specific workflow role inside a larger application.

In [ ]:
SYSTEM_PROMPT = "You are a senior engineer doing requirements review. Respond ONLY with clarifying questions that surface missing requirements. Do not propose a solution."
print(get_completion("Build me a rate limiter.", SYSTEM_PROMPT))

<a id="exercises"></a>
## Exercises

### Exercise 1.1: Count to three

**Task**: Edit `PROMPT` so that Claude's response contains the numbers 1, 2, and 3.

**Success criteria**: the response text includes all of `"1"`, `"2"`, and `"3"`.

In [ ]:
from lib.grading import includes_all, grade

PROMPT = "[Replace this text]"

response = get_completion(PROMPT)

def grade_exercise(text):
    return includes_all(text, ["1", "2", "3"])

grade(response, grade_exercise(response))

In [ ]:
from hints import exercise_1_1_hint
print(exercise_1_1_hint)

### Exercise 1.2: Terse TODO answers

**Scenario**: you're building a CLI tool that surfaces quick engineering guidance. You want answers that are terse and action oriented, formatted as a `# TODO` style code comment, not prose.

**Task**: Edit `SYSTEM_PROMPT` so that Claude answers as a terse senior engineer who replies only with a `# TODO` style code comment.

**Success criteria**: the response contains `"TODO"`.

In [ ]:
from lib.grading import includes_any, grade

SYSTEM_PROMPT = "[Replace this text]"
PROMPT = "How should I handle a failed database connection on startup?"

response = get_completion(PROMPT, SYSTEM_PROMPT)

def grade_exercise(text):
    return includes_any(text, ["TODO"])

grade(response, grade_exercise(response))

In [ ]:
from hints import exercise_1_2_hint
print(exercise_1_2_hint)

## Common mistakes

* **Ending on an `"assistant"` turn**: on current models the API rejects a `messages` list whose last element has `"role": "assistant"` (the old prefill pattern). Always end with the user turn you want answered.
* **Sloppy conversation state**: starting with an `assistant` turn or appending two `user` turns in a row no longer raises an error, because the API tolerates and merges them. That makes these bugs silent. Keep your history strictly alternating, starting with `user`.
* **Putting constraints in the user turn instead of `system`**: rules buried in the user message can be overridden or forgotten partway through the conversation. Stable constraints belong in the system prompt where they persist across all turns.

**Reflect**: when would you put a constraint in the system prompt vs the user message? Consider what changes from conversation to conversation vs what is fixed for all users of your application.

<a id="example-playground"></a>
## Example Playground

Use the cell below to experiment freely. Swap in any prompt. The graders are not active here.

In [ ]:
print(get_completion("Explain the CAP theorem in two sentences."))